
# HMS – Harmful Brain Activity Classification: baseline and extensions

This notebook starts from the course baseline (88 spectrogram features, logistic regression and gradient boosting) and extends it step by step. **Sections 1, 2 and 5 are the baseline; sections 3, 4, 6, 7 and 8 are my additions.** The same patient-grouped cross-validation and the same KL metric are used for every model, so results are comparable within one `MODE`.

| Section | Content | Origin |
|---|---|---|
| 0 | Settings, imports, data paths | baseline (+ `FEATURE_SET`) |
| 1 | Labels as vote distributions | baseline (+ EEG offsets) |
| 2 | Spectrogram features (88 per window) | baseline |
| 3 | Multi-scale windows, 50 s layer, edge features | mine, idea from Deotte's Grad-CAM thread |
| 4 | Raw EEG features (regional spectra, filtering) | mine, formula from Deotte, filters from the "Mega Model" notebook |
| 5 | Soft-label training, KL metric, baseline models | baseline |
| 6 | Extra models: ExtraTrees, MLP, small gradient boosting | mine |
| 7 | Cross-validation of ExtraTrees | mine |
| 8 | Small CNN with KL loss | mine |
| 9 | Model choice and submission | baseline (+ extensions) |

**Settings** (next cell): `MODE = "full"` uses all training spectrograms (about 10 minutes on a Kaggle CPU);
`MODE = "quick"` uses a random 3,000 for a fast check. `RUN_CV = False` skips cross-validation.

Runs on CPU only; internet must be off for Kaggle submissions.

In [1]:
print("Hello world")
import os
import time
import warnings
from mne.filter import filter_data, notch_filter
from glob import glob


import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Some spectrograms contain frequency bands that are entirely NaN; numpy warns about the
# resulting empty means. These cases are handled explicitly below, so the warnings are muted.
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ----------------------------------------------------------------------------- settings
MODE = "full"      # "full": all training spectrograms | "quick": a random subset for a fast check
RUN_CV = True      # True: run 5-fold cross-validation, report CV scores and pick the better model
MAX_SPECS = {"full": None, "quick": 3000}[MODE]   # number of training spectrograms to read (None = all)
N_FOLDS = 5        # number of cross-validation folds
N_JOBS = -1        # number of CPU cores used to read spectrograms in parallel (-1 = all cores)
SEED = 42          # random seed for subsampling and for the gradient-boosting model
FEATURE_SET = "eeg" # "spec" = alleen spectrogram | "eeg" = alleen ruwe EEG | "both" = beide
RUN_CNN = True      # zet op True voor de CNN-experimenten
# ----------------------------------------------------------------------------- data location
# Kaggle mounts the competition data at different paths depending on the notebook environment.
# HMS_DATA can be set to run the notebook elsewhere (e.g. on a local copy of the data).
CANDIDATES = ["/kaggle/input/competitions/hms-harmful-brain-activity-classification",
              "/kaggle/input/hms-harmful-brain-activity-classification",
              r"C:\MachineLearning opdracht"]                                                                    #nu kan ik runnen op kaggle en lokaal
DATA = next((p for p in CANDIDATES if os.path.exists(os.path.join(p, "train.csv"))), None)
if DATA is None:
    raise FileNotFoundError("train.csv niet gevonden in een van de paden in CANDIDATES.")
print("DATA =", DATA)

# ----------------------------------------------------------------------------- constants
# Vote columns in train.csv, in the order required by the submission file.
CLASSES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
# The official spectrograms average the bipolar EEG channels into four scalp regions:
# LL = left lateral, RL = right lateral, LP = left parasagittal, RP = right parasagittal.
REGIONS = ["LL", "RL", "LP", "RP"]
# Classical EEG frequency bands in Hz (the spectrograms cover roughly 0.5-20 Hz).
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 20)}

Hello world
DATA = /kaggle/input/competitions/hms-harmful-brain-activity-classification



## 1. Labels: one row per EEG, votes normalised to a probability distribution

`train.csv` has several rows (labelled sub-segments) per `eeg_id`. We sum the votes over these rows, divide by the total number of votes, and keep the range of label offsets to locate the window in the spectrogram and in the raw EEG. *My addition:* `eeg_center`, the middle of the labelled 50 s EEG segment (offset + 25 s), used in section 4.

In [2]:
train = pd.read_csv(f"{DATA}/train.csv")

# One row per EEG: spectrogram id, first/last label offset (seconds), patient id and summed votes per class.
agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"],
       "eeg_label_offset_seconds": ["min", "max"], "patient_id": "first"}
agg.update({c: "sum" for c in CLASSES})
df = train.groupby("eeg_id").agg(agg)
df.columns = ["spectrogram_id", "off_min", "off_max", "eeg_off_min", "eeg_off_max", "patient_id"] + CLASSES

# Convert vote counts into a probability distribution (each row sums to 1).
df["num_votes"] = df[CLASSES].sum(axis=1)
df[CLASSES] = df[CLASSES].values / df["num_votes"].values[:, None]
df = df.reset_index()
# Het 50 s-fragment begint op de offset; het label hoort bij de centrale 10 s (dus +25 s).
df["eeg_center"] = (df["eeg_off_min"] + df["eeg_off_max"]) / 2 + 25

# Quick mode: keep only the EEGs of a random subset of spectrograms.
if MAX_SPECS:
    keep = pd.Series(df.spectrogram_id.unique()).sample(n=min(MAX_SPECS, df.spectrogram_id.nunique()),
                                                         random_state=SEED)
    df = df[df.spectrogram_id.isin(keep)].reset_index(drop=True)

print(df.shape, "EEGs from", df.patient_id.nunique(), "patients,", df.spectrogram_id.nunique(), "spectrograms")
df.head()

(17089, 15) EEGs from 1950 patients, 11138 spectrograms


,eeg_id,spectrogram_id,off_min,off_max,eeg_off_min,eeg_off_max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,num_votes,eeg_center
0,568657,789577333,0.0,16.0,0.0,16.0,20654,0.0,0.000000,0.25,0.000000,0.166667,0.583333,48,33.0
1,582999,1552638400,0.0,38.0,0.0,38.0,20230,0.0,0.857143,0.00,0.071429,0.000000,0.071429,154,44.0
2,642382,14960202,1008.0,1032.0,0.0,24.0,5955,0.0,0.000000,0.00,0.000000,0.000000,1.000000,2,37.0
3,751790,618728447,908.0,908.0,0.0,0.0,38549,0.0,0.000000,1.00,0.000000,0.000000,0.000000,1,25.0
4,778705,52296320,0.0,0.0,0.0,0.0,40955,0.0,0.000000,0.00,0.000000,0.000000,1.000000,2,25.0


## 2. Spectrogram features (baseline)

Each spectrogram file has one row every 2 seconds and 400 power columns named `<region>_<frequency>` (e.g. `LL_1.17`). `window_features` reduces one window to 88 numbers:

| Feature group | Count |
|---|---|
| Log band power per region and band: mean, std, min, max over time | 4 regions × 4 bands × 4 = 64 |
| Relative band power (band power / total power of the region) | 4 × 4 = 16 |
| Left–right asymmetry of mean log band power | 2 × 4 = 8 |

In section 3 this function is applied to several windows, so the number of features grows beyond 88.

In [3]:
def band_columns(columns):
    """Return {(region, band): [column indices]} for the 400 spectrogram columns like 'LL_1.17'."""
    out = {}
    for r in REGIONS:
        for b, (lo, hi) in BANDS.items():
            out[(r, b)] = [i for i, c in enumerate(columns)
                           if c.startswith(r + "_") and lo <= float(c.split("_")[1]) < hi]
    return out


def window_features(spec, cols):
    """Compute the 88 features for one 10-minute window.

    spec: float array of shape (time steps, 400 power columns).
    cols: output of band_columns().
    """
    feats = {}
    total = {r: 0.0 for r in REGIONS}   # total power per region, for relative band power
    band_power = {}
    for (r, b), idx in cols.items():
        p = np.nanmean(spec[:, idx], axis=1)                     # mean power in the band, per time step
        lp = np.log1p(np.nan_to_num(p, nan=0.0).clip(min=0))     # log scale; missing values -> 0
        feats[f"{r}_{b}_mean"] = lp.mean()
        feats[f"{r}_{b}_std"] = lp.std()
        feats[f"{r}_{b}_min"] = lp.min()
        feats[f"{r}_{b}_max"] = lp.max()
        band_power[(r, b)] = np.nanmean(p) if np.isfinite(p).any() else 0.0
        total[r] += band_power[(r, b)]
    for (r, b), v in band_power.items():                         # share of the region's total power
        feats[f"{r}_{b}_rel"] = v / (total[r] + 1e-6)
    for b in BANDS:                                              # left minus right hemisphere
        feats[f"asym_lat_{b}"] = feats[f"LL_{b}_mean"] - feats[f"RL_{b}_mean"]
        feats[f"asym_par_{b}"] = feats[f"LP_{b}_mean"] - feats[f"RP_{b}_mean"]
    return feats



## 3. Multi-scale windows, 50 s layer and edge features (my addition)

The label applies to the central 10 s, but the baseline averages over 10 minutes, which dilutes the labelled segment. `multiscale_features` applies `window_features` to windows centred on the label (1 row = 2 s):

| Layer | Half-width (rows) | Window | Purpose |
|---|---|---|---|
| `c10` | 2 | ~10 s | the labelled segment |
| `c50s` | 12 | ~50 s | length of the EEG snippet the experts viewed |
| `c60` | 15 | ~1 min | short-term context |
| `full` | all | 10 min | baseline window |

Added on top: **difference features** (`c10` minus each larger layer) and **edge features** (band means 12–24 s before and after the centre, and their average minus the centre). The edge features follow the Grad-CAM analysis by Deotte, where the CNN attended to the edges of the 50 s window for some classes. Result: 448 features. The 50 s layer and the edge features were added together, so their separate effects are not measured.

In [4]:
# halve breedte in rijen (1 rij = 2 s): c10 = 10 s, c50s = 50 s, c60 = 1 min
SCALES = {"c10": 2, "c50s": 12, "c60": 15, "full": None}


def mean_feats(sub, cols):
    """Alleen de 16 band-gemiddelden (regio x band), voor de randvensters."""
    if len(sub) == 0:
        return None
    f = window_features(sub, cols)
    return {k: v for k, v in f.items() if k.endswith("_mean") and k.split("_")[0] in REGIONS}


def multiscale_features(seg, cols):
    mid = len(seg) // 2
    feats = {}

    # 1. Lagen rond het midden
    for name, half in SCALES.items():
        sub = seg if half is None else seg[max(0, mid - half): mid + half + 1]
        if len(sub) == 0:
            sub = seg
        for k, v in window_features(sub, cols).items():
            feats[f"{name}_{k}"] = v

    # 2. Centrum minus omgeving, voor elke laag groter dan c10
    for name in SCALES:
        if name == "c10":
            continue
        for r in REGIONS:
            for b in BANDS:
                feats[f"diff_c10_{name}_{r}_{b}"] = feats[f"c10_{r}_{b}_mean"] - feats[f"{name}_{r}_{b}_mean"]

    # 3. Randen van het 50 s-fragment: ongeveer 12-24 s voor en na het midden
    left = mean_feats(seg[max(0, mid - 12): max(0, mid - 5)], cols)
    right = mean_feats(seg[mid + 6: mid + 13], cols)
    for r in REGIONS:
        for b in BANDS:
            key = f"{r}_{b}_mean"
            l = left[key] if left else feats[f"full_{key}"]
            rt = right[key] if right else feats[f"full_{key}"]
            feats[f"edgeL_{r}_{b}"] = l
            feats[f"edgeR_{r}_{b}"] = rt
            feats[f"edge_vs_c10_{r}_{b}"] = (l + rt) / 2 - feats[f"c10_{key}"]
    return feats


### Building the features

`spectrogram_features` locates the window in each spectrogram file and calls `multiscale_features`. `build_features` reads each file once, in parallel, for all EEGs that use it. The test set uses the middle 10 minutes of each (exactly 10-minute) test spectrogram.

In [5]:



def spectrogram_features(path, windows, is_test):
    """Read one spectrogram file and compute features for every EEG that uses it.

    windows: list of (eeg_id, off_min, off_max); offsets are the label positions in seconds.
    """
    spec = pd.read_parquet(path)
    cols = band_columns(list(spec.columns[1:]))
    spec = spec.iloc[:, 1:].to_numpy(dtype=np.float32)          # drop the 'time' column
    rows = []
    for eeg_id, off_min, off_max in windows:
        if is_test:
            # Test spectrograms are short and centred on the segment: take their middle 10 minutes.
            start = max(0, (len(spec) - 300) // 2)
        else:
            # Training spectrograms cover the whole recording. Start at the midpoint of the labelled
            # offsets; offsets are in seconds and rows are 2 s apart, hence the division by 4 (= /2 /2).
            start = int((off_min + off_max) // 4)
        feats = multiscale_features(spec[start:start + 300], cols)
        feats["eeg_id"] = eeg_id
        rows.append(feats)
    return rows


def build_features(meta, spec_dir, is_test=False):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(f"{spec_dir}/{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=N_JOBS, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


X_df = build_features(df, f"{DATA}/train_spectrograms")
df = df.merge(X_df, on="eeg_id")
FEATS = [c for c in X_df.columns if c != "eeg_id"]
print(len(FEATS), "features")

/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_49/1471651478.py:21: RuntimeWarning: Mean of empt

11138 spectrograms -> 17089 feature rows in 286s
448 features


## 4. Raw EEG features 

Following Deotte's EEG-to-spectrogram formula, each of the four regions is built from four bipolar channel pairs (e.g. LL: Fp1–F7, F7–T3, T3–T5, T5–O1). For each pair we compute a Welch spectrum and **average the four spectra** (averaging the signals first would drop the intermediate electrodes). Per region and band we take log power, relative power, line length, standard deviation and left–right asymmetry, on the full 50 s segment and on the central 10 s, plus their difference: 112 features. A 60 Hz notch filter and a 0.5–40 Hz band-pass filter are applied first, following the preprocessing in the "Mega Model" notebook. Set `FEATURE_SET = "spec"` to skip this step; it is not used by the CNN.

In [6]:
from scipy.signal import welch

FS = 200   # EEG: 200 samples per seconde

# Deotte's formule: per regio vier bipolaire paren. Per paar een spectrum, daarna het GEMIDDELDE
# van die vier spectra. Niet eerst de signalen optellen, want dan vallen de tussenliggende elektroden weg.
MONTAGE = {
    "LL": [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1")],
    "LP": [("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1")],
    "RP": [("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")],
    "RL": [("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2")],
}
PAIRS = [(a, b) for pairs in MONTAGE.values() for a, b in pairs]      # 16 paren, 4 per regio, in MONTAGE-volgorde
EEG_CHANNELS = sorted({c for a, b in PAIRS for c in (a, b)})
CH_IDX = {c: i for i, c in enumerate(EEG_CHANNELS)}


def eeg_window_features(x, prefix):
    """x: (samples, kanalen in volgorde van EEG_CHANNELS). Geeft de features voor één tijdvenster."""
    bip = np.stack([x[:, CH_IDX[a]] - x[:, CH_IDX[b]] for a, b in PAIRS], axis=1)       # (samples, 16)
    f, pxx = welch(bip, fs=FS, nperseg=min(400, len(bip)), axis=0)                        # 2 s-segmenten, 0.5 Hz
    feats, logpow = {}, {}
    for i, r in enumerate(MONTAGE):
        spec = pxx[:, 4 * i: 4 * i + 4].mean(axis=1)                                      # gemiddelde van de 4 spectra
        total = spec[(f >= 0.5) & (f < 20)].sum() + 1e-12
        for b, (lo, hi) in BANDS.items():
            m = (f >= lo) & (f < hi)
            logpow[(r, b)] = np.log10(spec[m].mean() + 1e-6)
            feats[f"{prefix}_{r}_{b}_logpow"] = logpow[(r, b)]
            feats[f"{prefix}_{r}_{b}_relpow"] = spec[m].sum() / total
        sig = bip[:, 4 * i: 4 * i + 4]
        feats[f"{prefix}_{r}_linelen"] = np.abs(np.diff(sig, axis=0)).mean()              # scherpte/pieken
        feats[f"{prefix}_{r}_std"] = sig.std()
    for b in BANDS:                                                                        # links minus rechts
        feats[f"{prefix}_asym_lat_{b}"] = logpow[("LL", b)] - logpow[("RL", b)]
        feats[f"{prefix}_asym_par_{b}"] = logpow[("LP", b)] - logpow[("RP", b)]
    return feats


def eeg_features(path, center_sec, eeg_id, is_test):
    x = pd.read_parquet(path, columns=EEG_CHANNELS).to_numpy(dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0)
    n50 = 50 * FS
    start = 0 if is_test else int(center_sec * FS) - n50 // 2        # test_eegs zijn precies 50 s
    start = int(np.clip(start, 0, max(0, len(x) - n50)))
    seg = x[start:start + n50]
    
    if len(seg) < n50:                                                # vangnet: te korte opname
        seg = np.pad(seg, ((0, n50 - len(seg)), (0, 0)))
    seg = notch_filter(seg.T.astype("float64"), FS, 60, verbose="ERROR")        # 60 Hz storing
    seg = filter_data(seg, FS, 0.5, 40, verbose="ERROR").T.astype("float32")     # 0,5-40 Hz
    mid = len(seg) // 2
    feats = eeg_window_features(seg, "eeg50")                                         # hele 50 s
    feats.update(eeg_window_features(seg[mid - 5 * FS: mid + 5 * FS], "eeg10"))       # centrale 10 s
    for k in [k for k in feats if k.startswith("eeg10_") and k.endswith("logpow")]:   # centrum minus context
        feats["eegdiff_" + k[len("eeg10_"):]] = feats[k] - feats[k.replace("eeg10_", "eeg50_")]
    feats["eeg_id"] = eeg_id
    return feats


def build_eeg_features(meta, eeg_dir, is_test=False):
    t0 = time.time()
    centers = [25.0] * len(meta) if is_test else meta["eeg_center"].tolist()
    jobs = [delayed(eeg_features)(f"{eeg_dir}/{e}.parquet", c, e, is_test)
            for e, c in zip(meta["eeg_id"], centers)]
    out = pd.DataFrame(Parallel(n_jobs=N_JOBS, batch_size=16)(jobs))
    print(f"{len(jobs)} EEG's -> {out.shape[1] - 1} features in {time.time() - t0:.0f}s")
    return out


SPEC_FEATS = list(FEATS)
if FEATURE_SET != "spec":
    E_df = build_eeg_features(df, f"{DATA}/train_eegs")
    df = df.merge(E_df, on="eeg_id")
    EEG_FEATS = [c for c in E_df.columns if c != "eeg_id"]
else:
    EEG_FEATS = []
FEATS = {"spec": SPEC_FEATS, "eeg": EEG_FEATS, "both": SPEC_FEATS + EEG_FEATS}[FEATURE_SET]
print(FEATURE_SET, "->", len(SPEC_FEATS), "spectrogram +", len(EEG_FEATS), "EEG =", len(FEATS), "features")

17089 EEG's -> 112 features in 414s
eeg -> 448 spectrogram + 112 EEG = 112 features


## 3. Soft-label training and patient-grouped cross-validation

**Soft labels in scikit-learn.** scikit-learn classifiers expect one class per sample. We therefore repeat each EEG
six times, once per class, and weight each copy by that class's vote fraction. Minimising the weighted
cross-entropy is then the same as minimising the KL divergence to the vote distribution.

**Cross-validation (CV).** The training data are split into 5 folds *by patient*. Each model is trained on 4 folds
and evaluated on the held-out fold; the out-of-fold predictions give an estimate of performance on unseen patients.
Grouping by patient matters: EEGs of the same patient are very similar, and a random split would overestimate
performance. We report the CV score on all EEGs and on EEGs with at least 10 votes (more reliable labels),
next to the score of a trivial model that always predicts the average vote distribution (the *prior*).



In [7]:
import time
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

DEV = "cuda" if torch.cuda.is_available() else "cpu"
CV_MODELS = ["hgb","et","hgb_small"]        # run when RUN_CV = True; add "et", "hgb_small", "knn", ... as needed
DEFAULT_BEST = "hgb_small"                  # used when RUN_CV = False


# ------------------------------------------------------------------ 1. metric and soft-label helpers
def kl_score(y_true, y_pred, eps=1e-15):
    """Mean KL divergence KL(y_true || y_pred) over samples, as in the competition metric."""
    p = np.clip(y_pred, eps, 1)
    p = p / p.sum(axis=1, keepdims=True)
    t = y_true
    return np.mean(np.sum(np.where(t > 0, t * np.log(np.where(t > 0, t, 1) / p), 0), axis=1))


def expand(X, Y):
    """Soft labels -> weighted hard labels: repeat each sample once per class, weight = vote fraction."""
    n, k = Y.shape
    keep = Y.reshape(-1) > 0
    return np.repeat(X, k, axis=0)[keep], np.tile(np.arange(k), n)[keep], Y.reshape(-1)[keep]


def to_six(model, P):
    """Make predict_proba always 6 columns, even if a class is missing in the training fold."""
    out = np.zeros((len(P), 6)); out[:, model.classes_] = P
    return out / out.sum(1, keepdims=True)


# ------------------------------------------------------------------ 2. wrappers: every model has fit(X, Y) and predict_proba(X)
class WeightedClf:
    """Classifier trained on soft labels via 6x repetition and sample_weight (as in the baseline)."""
    def __init__(self, model, step=None):          # step = name of the final step if `model` is a Pipeline
        self.model, self.step = model, step
    def fit(self, X, Y):
        Xe, ye, we = expand(X, Y)
        key = f"{self.step}__sample_weight" if self.step else "sample_weight"
        self.model.fit(Xe, ye, **{key: we})
        return self
    def predict_proba(self, X):
        return to_six(self.model, self.model.predict_proba(X))


class SoftRegressor:
    """Multi-output regressor on the vote fractions; output is clipped and renormalised to sum to 1."""
    def __init__(self, model):
        self.model = model
    def fit(self, X, Y):
        self.model.fit(X, Y)
        return self
    def predict_proba(self, X):
        p = np.clip(self.model.predict(X), 1e-6, None)
        return p / p.sum(axis=1, keepdims=True)


class HardLDA:
    """LDA has no sample_weight, so it only sees the most-voted class (loses soft-label information)."""
    def __init__(self, model):
        self.model = model
    def fit(self, X, Y):
        self.model.fit(X, Y.argmax(1))
        return self
    def predict_proba(self, X):
        return to_six(self.model, self.model.predict_proba(X))


class TorchMLP:
    """MLP trained by backpropagation with KL loss, weight decay (Tikhonov) and a choice of Adam or SGD."""
    def __init__(self, hidden=(128, 64), lr=1e-3, wd=1e-2, epochs=30, bs=64, opt="adam", track=None):
        self.h, self.lr, self.wd, self.epochs, self.bs, self.opt, self.track = hidden, lr, wd, epochs, bs, opt, track
        self.curve = []                            # (train loss, validation KL) per epoch, for training curves
    def fit(self, X, Y):
        torch.manual_seed(SEED)
        self.sc = StandardScaler().fit(X)
        Xt = torch.from_numpy(self.sc.transform(X).astype(np.float32))
        Yt = torch.from_numpy(Y.astype(np.float32))
        layers, d = [], Xt.shape[1]
        for h in self.h:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.3)]
            d = h
        self.net = nn.Sequential(*layers, nn.Linear(d, 6)).to(DEV)
        P = self.net.parameters()
        o = (torch.optim.AdamW(P, lr=self.lr, weight_decay=self.wd) if self.opt == "adam"
             else torch.optim.SGD(P, lr=self.lr, momentum=0.9, weight_decay=self.wd))
        for _ in range(self.epochs):
            self.net.train()
            perm, tot = torch.randperm(len(Xt)), 0.0
            for i in range(0, len(perm), self.bs):
                b = perm[i:i + self.bs]
                loss = F.kl_div(F.log_softmax(self.net(Xt[b].to(DEV)), 1), Yt[b].to(DEV), reduction="batchmean")
                o.zero_grad(); loss.backward(); o.step()
                tot += loss.item() * len(b)
            val = kl_score(self.track[1], self.predict_proba(self.track[0])) if self.track else np.nan
            self.curve.append((tot / len(Xt), val))
        return self
    @torch.no_grad()
    def predict_proba(self, X):
        self.net.eval()
        x = torch.from_numpy(self.sc.transform(X).astype(np.float32)).to(DEV)
        return F.softmax(self.net(x), 1).cpu().numpy()


# ------------------------------------------------------------------ 3. model registry: one line per model
MODELS = {
    # baseline (course template)
    "logreg":    lambda: WeightedClf(make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)),
                                     "logisticregression"),
    "hgb":       lambda: WeightedClf(HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=31,
                                                                    l2_regularization=1.0, random_state=SEED)),
    # tree models on vote fractions / stronger regularisation
    "et":        lambda: SoftRegressor(ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.3,
                                                           n_jobs=-1, random_state=SEED)),
    "hgb_small": lambda: WeightedClf(HistGradientBoostingClassifier(learning_rate=0.03, max_iter=300, max_leaf_nodes=15,
                                                                    min_samples_leaf=100, l2_regularization=5.0,
                                                                    early_stopping=False, random_state=SEED)),
    # models from the course
    "mlp":       lambda: SoftRegressor(make_pipeline(StandardScaler(),
                                                     MLPRegressor(hidden_layer_sizes=(128, 64), alpha=1e-2,
                                                                  early_stopping=True, random_state=SEED))),
    "mlp_torch": lambda: TorchMLP(),
    "knn":       lambda: SoftRegressor(make_pipeline(StandardScaler(), PCA(n_components=30, random_state=SEED),
                                                     KNeighborsRegressor(n_neighbors=50, weights="distance"))),
    "pca_logreg": lambda: WeightedClf(make_pipeline(StandardScaler(), PCA(n_components=50, random_state=SEED),
                                                    LogisticRegression(C=0.1, max_iter=2000)), "logisticregression"),
    "kmeans_logreg": lambda: WeightedClf(make_pipeline(StandardScaler(), KMeans(n_clusters=30, n_init=3, random_state=SEED),
                                                       LogisticRegression(C=0.1, max_iter=2000)), "logisticregression"),
    "lda":       lambda: HardLDA(LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    "sgd":       lambda: WeightedClf(make_pipeline(StandardScaler(),
                                                   SGDClassifier(loss="log_loss", alpha=1e-3, max_iter=30, random_state=SEED)),
                                     "sgdclassifier"),
}


def fit(name, X, Y):
    """Fit model `name` on features X and soft labels Y."""
    return MODELS[name]().fit(X, Y)


# ------------------------------------------------------------------ 4. data arrays
X = np.nan_to_num(df[FEATS].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
Y = df[CLASSES].to_numpy()
groups = df["patient_id"].to_numpy()
prior = np.tile(Y.mean(axis=0), (len(Y), 1))     # trivial reference: always predict the mean distribution
hi = df["num_votes"].to_numpy() >= 10            # EEGs with at least 10 expert votes
folds = list(GroupKFold(n_splits=N_FOLDS).split(X, Y, groups))   # same folds for every model
oof = {}                                         # out-of-fold predictions per model


# ------------------------------------------------------------------ 5. cross-validation and results table
def run_cv(names):
    for name in names:
        t0 = time.time()
        oof[name] = np.zeros_like(Y)
        for i, (tr, va) in enumerate(folds):
            oof[name][va] = fit(name, X[tr], Y[tr]).predict_proba(X[va])
            print(f"  {name} fold {i + 1}/{len(folds)}: {time.time() - t0:.0f}s", flush=True)
        print(f"{name}: KL all {kl_score(Y, oof[name]):.4f} | >=10 votes {kl_score(Y[hi], oof[name][hi]):.4f}", flush=True)


def results_table():
    rows = [{"model": "prior (trivial)", "KL_all": kl_score(Y, prior),
             "KL_>=10votes": kl_score(Y[hi], prior[hi]), "std_folds": np.nan}]
    for name, p in oof.items():
        rows.append({"model": name, "KL_all": kl_score(Y, p), "KL_>=10votes": kl_score(Y[hi], p[hi]),
                     "std_folds": np.std([kl_score(Y[va], p[va]) for _, va in folds])})
    t = pd.DataFrame(rows).sort_values("KL_all").round(4).reset_index(drop=True)
    t.insert(1, "mode", MODE)
    t.insert(2, "n_features", X.shape[1])
    return t


if RUN_CV:
    run_cv(CV_MODELS)
    table = results_table()
    print(table.to_string())
    table.to_csv(f"results_{MODE}.csv", index=False)
    BEST = min(oof, key=lambda n: kl_score(Y, oof[n]))     # lower KL is better
else:
    BEST = DEFAULT_BEST
print("model for submission:", BEST)

  hgb fold 1/5: 10s
  hgb fold 2/5: 20s
  hgb fold 3/5: 30s
  hgb fold 4/5: 41s
  hgb fold 5/5: 50s
hgb: KL all 0.8842 | >=10 votes 0.8723
  et fold 1/5: 6s
  et fold 2/5: 13s
  et fold 3/5: 19s
  et fold 4/5: 25s
  et fold 5/5: 31s
et: KL all 0.9175 | >=10 votes 0.8888
  hgb_small fold 1/5: 19s
  hgb_small fold 2/5: 38s
  hgb_small fold 3/5: 57s
  hgb_small fold 4/5: 76s
  hgb_small fold 5/5: 95s
hgb_small: KL all 0.8797 | >=10 votes 0.8667
             model  mode  n_features  KL_all  KL_>=10votes  std_folds
0        hgb_small  full         112  0.8797        0.8667     0.0352
1              hgb  full         112  0.8842        0.8723     0.0416
2               et  full         112  0.9175        0.8888     0.0316
3  prior (trivial)  full         112  1.2559        1.2071        NaN
model for submission: hgb_small


ExtraTreesRegressor` and `MLPRegressor` are trained directly on the vote fractions (multi-output), so they need no 6× repetition and no `predict_proba`. The wrapper `SoftRegressor` clips their output at a small positive value and renormalises each row to sum to 1, so they fit the same interface as the classifiers. `hgb_small` is a more regularised gradient boosting model (15 leaves, `min_samples_leaf=100`, L2 = 5, no early stopping), because the labels are noisy.


Volgende code beslist welk model beste is

### Model choice

`BEST` is fixed to ExtraTrees, based on the quick-mode cross-validation (KL 1.007 versus 1.023 for `hgb_small`). 

## 7. Cross-validation of ExtraTrees

Out-of-fold KL of ExtraTrees on the selected features, with the same folds as all other models. We run this once per `FEATURE_SET` to compare spectrogram only, EEG only and both.

## 8. Small convolutional network with KL loss (my addition)

**Input.** The four regional spectrograms (LL, RL, LP, RP) as channels of one image: 100 frequency bins × 100 time rows (200 s around the label), log-transformed and standardised per region.

**Network.** Four blocks of 3×3 convolution, batch normalisation, ReLU and max-pooling; global average pooling; dropout 0.3; a linear layer with 6 outputs. Trained with a KL loss on the soft labels (AdamW, learning rate 1e-3, weight decay 1e-2, 8 epochs), and retrained for every fold, so the validation patients are never seen during training. Without pretraining or augmentation, this is a small learned-feature counterpart to the tree models.

In [ ]:
if RUN_CNN:
    import torch, torch.nn as nn, torch.nn.functional as F

    CNN_ROWS = 100            # centrale 100 rijen = 200 s (1 rij = 2 s); mijn keuze, zelf te variëren
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", DEV)
    
    
    def load_imgs(path, windows, is_test):
        """Per EEG een beeld (4 regio's x 100 frequenties x CNN_ROWS tijdstappen) uit één spectrogram-bestand."""
        spec = pd.read_parquet(path)
        cols = list(spec.columns[1:])
        arr = spec.iloc[:, 1:].to_numpy(dtype=np.float32)
        idx = [[i for i, c in enumerate(cols) if c.startswith(r + "_")] for r in REGIONS]
        out = []
        for eeg_id, off_min, off_max in windows:
            start = max(0, (len(arr) - 300) // 2) if is_test else int((off_min + off_max) // 4)   # zelfde als de baseline
            lo = start + 150 - CNN_ROWS // 2                                                       # midden van het 300-rijen venster
            seg = arr[lo: lo + CNN_ROWS]
            if len(seg) < CNN_ROWS:                                                                # vangnet: te kort spectrogram
                seg = np.pad(seg, ((0, CNN_ROWS - len(seg)), (0, 0)))
            img = np.stack([seg[:, ix].T for ix in idx])                                          # (4, 100, CNN_ROWS)
            img = np.log(np.clip(np.nan_to_num(img, nan=0.0), np.exp(-4), np.exp(8)))             # log-schaal, pieken begrensd
            img = (img - img.mean((1, 2), keepdims=True)) / (img.std((1, 2), keepdims=True) + 1e-6)
            out.append((eeg_id, img.astype(np.float16)))
        return out
    
    
    def build_imgs(meta, spec_dir, is_test=False):
        t0 = time.time()
        if is_test:
            meta = meta.assign(off_min=0.0, off_max=0.0)
        jobs = [delayed(load_imgs)(f"{spec_dir}/{sid}.parquet", list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
                for sid, g in meta.groupby("spectrogram_id")]
        res = Parallel(n_jobs=N_JOBS, batch_size=32)(jobs)
        d = {e: im for rows in res for e, im in rows}
        print(f"{len(d)} beelden in {time.time() - t0:.0f}s")
        return d
    
    
    imgs = build_imgs(df, f"{DATA}/train_spectrograms")
    IMG = np.stack([imgs[e] for e in df["eeg_id"]])      # zelfde volgorde als df, dus als Y en groups
    del imgs
    print(IMG.shape, IMG.dtype)

    

device: cuda


Beelden maken uit spec


### Training and cross-validation of the CNN

Same patient-grouped folds as the other models. Results from a single seed vary by about 0.04 between runs, so differences of that size are noise; average several seeds before comparing.

In [ ]:
if RUN_CNN:
    torch.manual_seed(SEED); np.random.seed(SEED)
    
    
    class SmallCNN(nn.Module):
        def __init__(self):
            super().__init__()
            def blk(i, o):
                return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
            self.f = nn.Sequential(blk(4, 16), blk(16, 32), blk(32, 64), blk(64, 64))
            self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(64, 6))
    
        def forward(self, x):
            return self.head(self.f(x).mean((2, 3)))          # global average pooling, dan 6 logits
    
    
    def train_cnn(Xtr, Ytr, epochs=8, bs=64, lr=1e-3):
        m = SmallCNN().to(DEV)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-2)
        Xt, Yt = torch.from_numpy(Xtr), torch.from_numpy(Ytr.astype(np.float32))
        for ep in range(epochs):
            m.train()
            perm = torch.randperm(len(Xt))
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]
                xb, yb = Xt[b].float().to(DEV), Yt[b].to(DEV)
                loss = F.kl_div(F.log_softmax(m(xb), 1), yb, reduction="batchmean")     # KL op de soft labels
                opt.zero_grad(); loss.backward(); opt.step()
        return m
    
    
    @torch.no_grad()
    def predict_cnn(m, X, bs=256):
        m.eval()
        return np.concatenate([F.softmax(m(torch.from_numpy(X[i:i + bs]).float().to(DEV)), 1).cpu().numpy()
                               for i in range(0, len(X), bs)])
    
    
    oof_cnn = np.zeros_like(Y)
    for k, (tr, va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(IMG, Y, groups)):
        t0 = time.time()
        m = train_cnn(IMG[tr], Y[tr])
        oof_cnn[va] = predict_cnn(m, IMG[va])
        print(f"fold {k}: KL {kl_score(Y[va], oof_cnn[va]):.4f}  ({time.time() - t0:.0f}s)", flush=True)
    
    print(f"CNN | KL all {kl_score(Y, oof_cnn):.4f} | >=10 votes {kl_score(Y[hi], oof_cnn[hi]):.4f}")

## 9. Model choice and submission

The model with the lowest out-of-fold KL (ExtraTrees, the CNN, or an average of both) is refitted on all training data and predicts the hidden test set. The visible `test.csv` contains one example; Kaggle re-runs the notebook with the full hidden test set, so this cell works for any number of test EEGs.

In [ ]:
import matplotlib.pyplot as plt
tr, va = folds[0]
plt.figure(figsize=(11, 4))
for k, (lr, opt) in enumerate([(1e-4, "adam"), (1e-3, "adam"), (1e-2, "adam"), (1e-2, "sgd")]):
    m = TorchMLP(lr=lr, opt=opt, epochs=40, track=(X[va], Y[va])).fit(X[tr], Y[tr])
    c = np.array(m.curve)
    plt.subplot(1, 2, 1); plt.plot(c[:, 0], label=f"{opt} lr={lr}")
    plt.subplot(1, 2, 2); plt.plot(c[:, 1], label=f"{opt} lr={lr}")
plt.subplot(1, 2, 1); plt.title("train-KL per epoch"); plt.xlabel("epoch"); plt.legend()
plt.subplot(1, 2, 2); plt.title("validatie-KL per epoch (fold 0)"); plt.xlabel("epoch"); plt.legend()
plt.tight_layout(); plt.show()

In [11]:
# ------------------------------------------------------------------ 1. what goes into the submission
# BEST = model with the lowest out-of-fold KL among CV_MODELS (chosen in section 5).
# If the CNN was cross-validated as well (RUN_CNN = True), we test a weighted average
#     w * BEST + (1 - w) * CNN
# on the same out-of-fold predictions and use it only if it is clearly better than BEST alone.
USE_CNN, W = False, 1.0                       # W = weight of BEST in the blend
if RUN_CNN and RUN_CV and "oof_cnn" in globals() and BEST in oof:
    ws = np.linspace(0, 1, 21)
    kls = [kl_score(Y, w * oof[BEST] + (1 - w) * oof_cnn) for w in ws]
    W, kl_blend, kl_best = float(ws[int(np.argmin(kls))]), min(kls), kl_score(Y, oof[BEST])
    USE_CNN = W < 1.0 and kl_blend < kl_best - 0.002        # require a gain larger than the noise level
    print(f"{BEST} alone: {kl_best:.4f} | best blend (w={W:.2f} {BEST} + {1 - W:.2f} CNN): {kl_blend:.4f} "
          f"-> {'blend' if USE_CNN else BEST + ' alone'} goes into the submission")
elif RUN_CNN:
    print("RUN_CNN is on, but the CNN has no cross-validation (RUN_CV / oof_cnn missing): CNN not used.")
print("model for submission:", BEST, "+ CNN" if USE_CNN else "")

# ------------------------------------------------------------------ 2. refit on all training data
final = fit(BEST, X, Y)

# ------------------------------------------------------------------ 3. test features
test = pd.read_csv(f"{DATA}/test.csv")
T_df = build_features(test, f"{DATA}/test_spectrograms", is_test=True)
T_df = test[["eeg_id"]].merge(T_df, on="eeg_id", how="left")   # keep the order of test.csv
if FEATURE_SET != "spec":
    TE_df = build_eeg_features(test, f"{DATA}/test_eegs", is_test=True)
    T_df = T_df.merge(TE_df, on="eeg_id", how="left")
XT = np.nan_to_num(T_df[FEATS].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

# ------------------------------------------------------------------ 4. predictions (optionally blended with the CNN)
P = final.predict_proba(XT)
if USE_CNN:
    timgs = build_imgs(test, f"{DATA}/test_spectrograms", is_test=True)
    IMG_T = np.stack([timgs[e] for e in T_df["eeg_id"]])        # same order as T_df / test.csv
    cnn_final = train_cnn(IMG, Y)                               # CNN refitted on all training data
    P = W * P + (1 - W) * predict_cnn(cnn_final, IMG_T)
    P = P / P.sum(axis=1, keepdims=True)

# Submission format: eeg_id plus one predicted probability per class (each row sums to 1).
sub = pd.DataFrame({"eeg_id": T_df["eeg_id"]})
sub[CLASSES] = P
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.head()

1 spectrograms -> 1 feature rows in 0s
1 EEG's -> 112 features in 0s
1 beelden in 0s
(1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.082743,0.04508,0.001555,0.119742,0.046998,0.703882
